In [1]:
import numpy as np
import pandas as pd
import os
import h5py
import json
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# LNN
import torch
import torch.nn as nn
from LiqudNet.liquidnet.main import LiquidNet
from torch.utils.data import random_split, DataLoader

#KAN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, TensorDataset
from tqdm.auto import tqdm
import numpy as np

import torch
import torch.nn.functional as F
import math

from torchsummary import summary

# Preprocessing
import re
import nltk
import spacy
import fasttext
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [2]:
# Already created so no need to run this

hdf5_path = "../Datasets/Homogeneous/Embeddings/FNN_Character_Embeddings.hdf5"

# Get sorted batch files
batch_files = sorted(os.listdir('../Datasets/Homogeneous/Embeddings/FNN_CHAR/'), key = lambda x: int(x.split('_')[-1].split('.npy')[0]))

# Determine total number of samples
total_samples = 21684

# Open an HDF5 file for writing
with h5py.File(hdf5_path, "w") as f:
    # Define dataset shape (total_samples, 2000, 128), stored as float16
    dset = f.create_dataset("Character_Embeddings", shape = (total_samples, 2000, 128), dtype = np.float16)
 
    # Write each batch into the dataset
    index = 0
    for file in batch_files:
        batch_path = os.path.join('../Datasets/Homogeneous/Embeddings/FNN_CHAR/', file)
        batch_data = np.load(batch_path).astype(np.float16)  # Convert to float16 to save memory
        batch_size = batch_data.shape[0]

        dset[index : index + batch_size] = batch_data  # Store batch in HDF5
        index += batch_size
        
        del batch_data

print("HDF5 file created successfully!")

HDF5 file created successfully!


In [3]:
# Already created so no need to run this

hdf5_path = "../Datasets/Homogeneous/Embeddings/Liar-Plus_Character_Embeddings.hdf5"

# Get sorted batch files
batch_files = sorted(os.listdir('../Datasets/Homogeneous/Embeddings/LIAR-PLUS_CHAR/'), key = lambda x: int(x.split('_')[-1].split('.npy')[0]))

# Determine total number of samples
total_samples = 8048

# Open an HDF5 file for writing
with h5py.File(hdf5_path, "w") as f:
    # Define dataset shape (total_samples, 2000, 128), stored as float16
    dset = f.create_dataset("Character_Embeddings", shape = (total_samples, 2000, 128), dtype = np.float16)
 
    # Write each batch into the dataset
    index = 0
    for file in batch_files:
        batch_path = os.path.join('../Datasets/Homogeneous/Embeddings/LIAR-PLUS_CHAR/', file)
        batch_data = np.load(batch_path).astype(np.float16)  # Convert to float16 to save memory
        batch_size = batch_data.shape[0]

        dset[index : index + batch_size] = batch_data  # Store batch in HDF5
        index += batch_size
        
        del batch_data

print("HDF5 file created successfully!")

HDF5 file created successfully!


In [4]:
# Already created so no need to run this

hdf5_path = "../Datasets/Homogeneous/Embeddings/Politifact_Character_Embeddings.hdf5"

# Get sorted batch files
batch_files = sorted(os.listdir('../Datasets/Homogeneous/Embeddings/POLITIFACT_CHAR/'), key = lambda x: int(x.split('_')[-1].split('.npy')[0]))

# Determine total number of samples
total_samples = 17540

# Open an HDF5 file for writing
with h5py.File(hdf5_path, "w") as f:
    # Define dataset shape (total_samples, 2000, 128), stored as float16
    dset = f.create_dataset("Character_Embeddings", shape = (total_samples, 2000, 128), dtype = np.float16)
 
    # Write each batch into the dataset
    index = 0
    for file in batch_files:
        batch_path = os.path.join('../Datasets/Homogeneous/Embeddings/POLITIFACT_CHAR/', file)
        batch_data = np.load(batch_path).astype(np.float16)  # Convert to float16 to save memory
        batch_size = batch_data.shape[0]

        dset[index : index + batch_size] = batch_data  # Store batch in HDF5
        index += batch_size
        
        del batch_data

print("HDF5 file created successfully!")

HDF5 file created successfully!


In [53]:
# Already created so no need to run this

hdf5_path = "../Datasets/Homogeneous/Embeddings/ISOT_Character_Embeddings.hdf5"

# Get sorted batch files
batch_files = sorted(os.listdir('../Datasets/Homogeneous/Embeddings/ISOT_CHAR/'), key = lambda x: int(x.split('_')[-1].split('.npy')[0]))

# Determine total number of samples
total_samples = 38591

# Open an HDF5 file for writing
with h5py.File(hdf5_path, "w") as f:
    # Define dataset shape (total_samples, 2000, 128), stored as float16
    dset = f.create_dataset("Character_Embeddings", shape = (total_samples, 2000, 128), dtype = np.float16)
 
    # Write each batch into the dataset
    index = 0
    for file in batch_files:
        batch_path = os.path.join('../Datasets/Homogeneous/Embeddings/ISOT_CHAR/', file)
        batch_data = np.load(batch_path).astype(np.float16)  # Convert to float16 to save memory
        batch_size = batch_data.shape[0]

        dset[index : index + batch_size] = batch_data  # Store batch in HDF5
        index += batch_size
        
        del batch_data

print("HDF5 file created successfully!")

HDF5 file created successfully!


In [4]:
class FakeNewsDataset(Dataset):
    def __init__(self, word_embeddings_file, positional_embeddings_file, character_embeddings_hdf5, labels_file = None):
        self.word_embeddings = np.load(word_embeddings_file)
        self.positional_embeddings = np.load(positional_embeddings_file)

        # Open HDF5 file in read mode
#         self.h5_file = h5py.File(character_embeddings_hdf5, "r")
#         self.character_embeddings = self.h5_file["Character_Embeddings"]
        self.character_embeddings = np.load(character_embeddings_hdf5)

        # Load labels if available
        if labels_file:
            self.labels = pd.read_csv(labels_file, header = None, usecols = [0]).values[1:]
            self.labels = np.where(self.labels == 'fake', 1, 0)
        else:
            self.labels = None

    def __len__(self):
        return len(self.word_embeddings)  # Assuming word embeddings match character embeddings

    def __getitem__(self, idx):
        word_embedding = torch.tensor(self.word_embeddings[idx], dtype = torch.float32)
        positional_embedding = torch.tensor(self.positional_embeddings[idx], dtype = torch.float32)
        character_embedding = torch.tensor(self.character_embeddings[idx], dtype = torch.float32)  # Efficiently read from HDF5

        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype = torch.long)
            return word_embedding, positional_embedding, character_embedding, label[0]
        else:
            return word_embedding, positional_embedding, character_embedding

In [5]:
# Path to the embeddings and labels
word_embeddings_file = '../Datasets/Homogeneous/Embeddings/FNC_WORD_EMBED.npy'
positional_embeddings_file = '../Datasets/Homogeneous/Embeddings/FNC_POS_EMBED.npy'
character_embeddings_hdf5 = '../Datasets/Homogeneous/Embeddings/FNC_char_level_embeddings-R.npy'
labels_file = '../Datasets/Homogeneous/Individual/FNC_P_2.csv'  # CSV file with a 'type' column for real/fake labels
# Initialize the dataset with both embeddings and labels
dataset = FakeNewsDataset(word_embeddings_file, positional_embeddings_file, character_embeddings_hdf5, labels_file)

In [6]:
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True, num_workers = 0)

In [7]:
for word_embeddings, positional_embeddings, character_embeddings, labels in dataloader:
    print("Batch:", word_embeddings.shape, positional_embeddings.shape, character_embeddings.shape, labels.shape)
    break

Batch: torch.Size([32, 120, 128]) torch.Size([32, 120, 128]) torch.Size([32, 2000, 128]) torch.Size([32])


In [14]:
class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size = 5,
        spline_order = 3,
        scale_noise = 0.1,
        scale_base = 1.0,
        scale_spline = 1.0,
        enable_standalone_scale_spline = True,
        base_activation = torch.nn.SiLU,
        grid_eps = 0.02,
        grid_range = [-1, 1],
    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a = math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (
                    torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                    - 1 / 2
                )
                * self.scale_noise
                / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order : -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                # torch.nn.init.constant_(self.spline_scaler, self.scale_spline)
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a = math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        """
        Compute the B-spline bases for the given input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).

        Returns:
            torch.Tensor: B-spline bases tensor of shape (batch_size, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features

        grid: torch.Tensor = (
            self.grid
        )  # (in_features, grid_size + 2 * spline_order + 1)
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - grid[:, : -(k + 1)])
                / (grid[:, k:-1] - grid[:, : -(k + 1)])
                * bases[:, :, :-1]
            ) + (
                (grid[:, k + 1 :] - x)
                / (grid[:, k + 1 :] - grid[:, 1:(-k)])
                * bases[:, :, 1:]
            )

        assert bases.size() == (
            x.size(0),
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        """
        Compute the coefficients of the curve that interpolates the given points.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).
            y (torch.Tensor): Output tensor of shape (batch_size, in_features, out_features).

        Returns:
            torch.Tensor: Coefficients tensor of shape (out_features, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features
        assert y.size() == (x.size(0), self.in_features, self.out_features)

        A = self.b_splines(x).transpose(
            0, 1
        )  # (in_features, batch_size, grid_size + spline_order)
        B = y.transpose(0, 1)  # (in_features, batch_size, out_features)
        solution = torch.linalg.lstsq(
            A, B
        ).solution  # (in_features, grid_size + spline_order, out_features)
        result = solution.permute(
            2, 0, 1
        )  # (out_features, in_features, grid_size + spline_order)

        assert result.size() == (
            self.out_features,
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        return self.spline_weight * (
            self.spline_scaler.unsqueeze(-1)
            if self.enable_standalone_scale_spline
            else 1.0
        )

    def forward(self, x: torch.Tensor):
        assert x.size(-1) == self.in_features
        original_shape = x.shape
        x = x.reshape(-1, self.in_features)

        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        output = base_output + spline_output

        output = output.reshape(*original_shape[:-1], self.out_features)
        return output

    @torch.no_grad()
    def update_grid(self, x: torch.Tensor, margin = 0.01):
        assert x.dim() == 2 and x.size(1) == self.in_features
        batch = x.size(0)

        splines = self.b_splines(x)  # (batch, in, coeff)
        splines = splines.permute(1, 0, 2)  # (in, batch, coeff)
        orig_coeff = self.scaled_spline_weight  # (out, in, coeff)
        orig_coeff = orig_coeff.permute(1, 2, 0)  # (in, coeff, out)
        unreduced_spline_output = torch.bmm(splines, orig_coeff)  # (in, batch, out)
        unreduced_spline_output = unreduced_spline_output.permute(
            1, 0, 2
        )  # (batch, in, out)

        # sort each channel individually to collect data distribution
        x_sorted = torch.sort(x, dim = 0)[0]
        grid_adaptive = x_sorted[
            torch.linspace(
                0, batch - 1, self.grid_size + 1, dtype=torch.int64, device = x.device
            )
        ]

        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (
            torch.arange(
                self.grid_size + 1, dtype = torch.float32, device=x.device
            ).unsqueeze(1)
            * uniform_step
            + x_sorted[0]
            - margin
        )

        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.concatenate(
            [
                grid[:1]
                - uniform_step
                * torch.arange(self.spline_order, 0, -1, device = x.device).unsqueeze(1),
                grid,
                grid[-1:]
                + uniform_step
                * torch.arange(1, self.spline_order + 1, device = x.device).unsqueeze(1),
            ],
            dim = 0,
        )

        self.grid.copy_(grid.T)
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        """
        Compute the regularization loss.

        This is a dumb simulation of the original L1 regularization as stated in the
        paper, since the original one requires computing absolutes and entropy from the
        expanded (batch, in_features, out_features) intermediate tensor, which is hidden
        behind the F.linear function if we want an memory efficient implementation.

        The L1 regularization is now computed as mean absolute value of the spline
        weights. The authors implementation also includes this term in addition to the
        sample-based regularization.
        """
        l1_fake = self.spline_weight.abs().mean(-1)
        regularization_loss_activation = l1_fake.sum()
        p = l1_fake / regularization_loss_activation
        regularization_loss_entropy = -torch.sum(p * p.log())
        return (
            regularize_activation * regularization_loss_activation
            + regularize_entropy * regularization_loss_entropy
        )

In [15]:
class KAN(torch.nn.Module):
    def __init__(
        self,
        layers_hidden,
        grid_size = 5,
        spline_order = 3,
        scale_noise = 0.1,
        scale_base = 1.0,
        scale_spline = 1.0,
        base_activation = torch.nn.SiLU,
        grid_eps = 0.02,
        grid_range = [-1, 1],
    ):
        super(KAN, self).__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order

        self.layers = torch.nn.ModuleList()
        for in_features, out_features in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(
                KANLinear(
                    in_features,
                    out_features,
                    grid_size = grid_size,
                    spline_order = spline_order,
                    scale_noise = scale_noise,
                    scale_base = scale_base,
                    scale_spline = scale_spline,
                    base_activation = base_activation,
                    grid_eps = grid_eps,
                    grid_range = grid_range,
                )
            )

    def forward(self, x: torch.Tensor, update_grid = False):
        for layer in self.layers:
            if update_grid:
                layer.update_grid(x)
            x = layer(x)
        return x

    def regularization_loss(self, regularize_activation = 1.0, regularize_entropy = 1.0):
        return sum(
            layer.regularization_loss(regularize_activation, regularize_entropy)
            for layer in self.layers
        )

In [16]:
class CharacterEmbeddingsReductionModel(nn.Module):
    def __init__(self, input_size = 128, hidden_units = 256, output_size = 1, num_heads = 4):
        super(CharacterEmbeddingsReductionModel, self).__init__()

        self.mish = nn.Mish()

        # CNN Block for Feature Extraction
        self.conv1 = nn.Conv1d(in_channels = input_size, out_channels = hidden_units // 2, kernel_size = 3, padding = 1).to("cuda" if torch.cuda.is_available() else "cpu")
        self.conv2 = nn.Conv1d(in_channels = hidden_units // 2, out_channels = hidden_units, kernel_size = 3, padding = 1).to("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_norm = nn.BatchNorm1d(hidden_units).to("cuda" if torch.cuda.is_available() else "cpu")

        # GRU Layers for Temporal Encoding
        self.gru_1 = nn.GRU(hidden_units, hidden_units // 4, num_layers = 1, bidirectional = True, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        self.norm_1 = nn.LayerNorm(hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")
        self.gru_2 = nn.GRU(hidden_units // 2, hidden_units // 64, num_layers = 1, bidirectional = False, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        self.norm_2 = nn.LayerNorm(hidden_units // 64).to("cuda" if torch.cuda.is_available() else "cpu")
        self.gru_3 = nn.GRU(hidden_units // 4, hidden_units // 16, num_layers = 1, bidirectional = True, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        self.norm_3 = nn.LayerNorm(hidden_units // 8).to("cuda" if torch.cuda.is_available() else "cpu")
        self.gru_4 = nn.GRU(hidden_units // 8, hidden_units // 64, num_layers = 1, bidirectional = False, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        self.norm_4 = nn.LayerNorm(hidden_units // 64).to("cuda" if torch.cuda.is_available() else "cpu")

        # Self-Attention Layer
        self.attention = nn.MultiheadAttention(embed_dim = hidden_units // 64, num_heads = num_heads, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        
        # Final Layers
        self.o_1 = nn.Linear(125 * hidden_units // 64, hidden_units).to("cuda" if torch.cuda.is_available() else "cpu")
        self.o_2 = nn.Linear(hidden_units, hidden_units // 2).to("cuda" if torch.cuda.is_available() else "cpu")
        self.o_3 = nn.Linear(hidden_units // 2, hidden_units // 4).to("cuda" if torch.cuda.is_available() else "cpu")
        self.o_4 = nn.Linear(hidden_units // 4, hidden_units // 8).to("cuda" if torch.cuda.is_available() else "cpu")
        self.o_5 = nn.Linear(hidden_units // 8, output_size).to("cuda" if torch.cuda.is_available() else "cpu")

        # Dropout for Regularization
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        batch_size = x.shape[0]

        # CNN Feature Extraction
        x = x.permute(0, 2, 1)  # Change to (batch, channels, seq_len)
        x = self.mish(self.conv1(x))
        x = self.mish(self.conv2(x))
        x = self.batch_norm(x)
        x = x.permute(0, 2, 1)  # Change back to (batch, seq_len, channels)

        # GRU Layers
        outputs, _ = self.gru_1(x)
        outputs = self.norm_1(outputs)
        outputs, _ = self.gru_2(outputs)
        outputs = self.norm_2(outputs)
        reduced_character_embeddings = outputs.reshape(batch_size, 125, -1)
        outputs, _ = self.gru_3(outputs.reshape(batch_size, 125, -1))
        outputs = self.norm_3(outputs)
        outputs, _ = self.gru_4(outputs)
        outputs = self.norm_4(outputs)

        # Self-Attention
        outputs, _ = self.attention(outputs, outputs, outputs)

        # Fully Connected Layers
        outputs = self.mish(self.o_1(outputs.reshape(batch_size, -1)))
        outputs = self.dropout(outputs)
        outputs = self.mish(self.o_2(outputs))
        outputs = self.dropout(outputs)
        outputs = self.mish(self.o_3(outputs))
        outputs = self.mish(self.o_4(outputs))
        outputs = self.o_5(outputs)

        return outputs, reduced_character_embeddings

In [17]:
class FakeNewsDetectionModel(nn.Module):
    def __init__(self, input_size, hidden_units, output_size):
        super(FakeNewsDetectionModel, self).__init__()
        # Word Embeddings Block + Positional Embeddings Block
        self.multihead_attn_wepe = nn.MultiheadAttention(input_size, 4, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        self.gru_1_wepe = nn.GRU(input_size, hidden_units // 4, num_layers = 1, bidirectional = True, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        self.gru_2_wepe = nn.GRU(hidden_units // 2, hidden_units // 16, num_layers = 1, bidirectional = False, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        
        # Character Embeddings Block
        self.conv1_ce = nn.Conv1d(hidden_units, hidden_units // 2, kernel_size = 3, stride = 1, padding = 1).to("cuda" if torch.cuda.is_available() else "cpu")
        self.conv2_ce = nn.Conv1d(hidden_units // 2, hidden_units // 4, kernel_size = 6, stride = 1, padding = 0).to("cuda" if torch.cuda.is_available() else "cpu")
        self.gru_1_ce = nn.GRU(hidden_units // 4, hidden_units // 16, num_layers = 1, bidirectional = True, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        self.gru_2_ce = nn.GRU(hidden_units // 8 , hidden_units // 16, num_layers = 1, bidirectional = False, batch_first = True).to("cuda" if torch.cuda.is_available() else "cpu")
        
        # Output and Shared Block
        self.fc = nn.Linear(120 * 2 * hidden_units // 16, hidden_units * 4).to("cuda" if torch.cuda.is_available() else "cpu")
        self.ltc_cell = LiquidNet(hidden_units * 2).to("cuda" if torch.cuda.is_available() else "cpu")  # LiquidNet cell
        self.kan_1 = KANLinear(hidden_units * 2, hidden_units, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu") # KAN
        self.kan_2 = KANLinear(hidden_units, hidden_units // 2, base_activation = nn.SiLU).to("cuda" if torch.cuda.is_available() else "cpu")
        self.o = nn.Linear(hidden_units // 2, output_size).to("cuda" if torch.cuda.is_available() else "cpu")
    
    def forward(self, x, y, z):
        # LiquidNet requires an initial hidden state
        batch_size = x.size(0)
        hidden_state = torch.zeros(batch_size, self.ltc_cell._num_units).to(x.device)
        
        # Word Embeddings Pass + Positional Embeddings Pass
        outputs = torch.add(x, y)
        outputs, _ = self.multihead_attn_wepe(outputs, outputs, outputs)
        outputs, _ = self.gru_1_wepe(outputs)
        outputs_wepe, _ = self.gru_2_wepe(outputs)
        
        # Character Embeddings Pass
        outputs = self.conv1_ce(z.reshape(batch_size, 64, 125))
        outputs = self.conv2_ce(outputs)
        outputs, _ = self.gru_1_ce(outputs.reshape(batch_size, 120, 16))
        outputs_ce, _ = self.gru_2_ce(outputs)
        
        # Pass through Output and Shared Block
        outputs = torch.cat((outputs_wepe, outputs_ce), dim = 2)
        outputs = self.fc(outputs.reshape(batch_size, 120 * hidden_units // 8))
        outputs, _ = self.ltc_cell(outputs, hidden_state)
        outputs = self.kan_1(outputs)
        outputs = self.kan_2(outputs)
        outputs = self.o(outputs)

        return outputs

In [18]:
def test(news_model, char_model, test_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    news_model = news_model.to(device)
    char_model = char_model.to(device)
    test_loss = 0.0
    criterion = nn.BCEWithLogitsLoss()
    y_true_test = []
    y_pred_test = []

    with torch.no_grad():
        for word_embeddings, pos_embeddings, char_embeddings, labels in tqdm(test_loader):
            word_embeddings, pos_embeddings, char_embeddings, labels = word_embeddings.to(device), pos_embeddings.to(device), char_embeddings.to(device), labels.to(device)
            
            _, reduced_character_embeddings = char_model(char_embeddings)
            outputs = news_model(word_embeddings.float(), pos_embeddings.float(), reduced_character_embeddings)
            loss = criterion(outputs, labels.unsqueeze(1).float())
            test_loss += loss.item()

            # Get predicted class indices
            outputs = nn.Sigmoid()(outputs)
            y_true_test.extend(labels.cpu().numpy())
            y_pred_test.extend((outputs > 0.5).detach().cpu().numpy())

    test_accuracy = accuracy_score(y_true_test, y_pred_test) * 100
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

In [19]:
input_size = 128  # Input size matches your embeddings
hidden_units_character = 256  # Number of units for LiquidNet
output_size = 1  # Binary classification (real/fake)
character_reduction_model = CharacterEmbeddingsReductionModel(input_size, hidden_units_character, output_size)

In [20]:
# Provide the model, input size (excluding batch size), and device
summary(character_reduction_model, input_size = (2000, input_size), device = "cuda" if torch.cuda.is_available() else "cpu")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv1d-1            [-1, 128, 2000]          49,280
              Mish-2            [-1, 128, 2000]               0
            Conv1d-3            [-1, 256, 2000]          98,560
              Mish-4            [-1, 256, 2000]               0
       BatchNorm1d-5            [-1, 256, 2000]             512
               GRU-6  [[-1, 2000, 128], [-1, 2, 64]]               0
         LayerNorm-7            [-1, 2000, 128]             256
               GRU-8  [[-1, 2000, 4], [-1, 2, 4]]               0
         LayerNorm-9              [-1, 2000, 4]               8
              GRU-10  [[-1, 125, 32], [-1, 2, 16]]               0
        LayerNorm-11              [-1, 125, 32]              64
              GRU-12  [[-1, 125, 4], [-1, 2, 4]]               0
        LayerNorm-13               [-1, 125, 4]               8
MultiheadAttention-14  [[-1,

In [21]:
# Initialize the model
input_size = 128  # Input size matches your embeddings
hidden_units = 64  # Number of units for LiquidNet
output_size = 1  # Binary classification (real/fake)
model = FakeNewsDetectionModel(input_size, hidden_units, output_size)

In [22]:
# Provide the model, input size (excluding batch size), and device
summary(model, input_size = [(120, input_size), (120, input_size), (125, hidden_units)], device = "cuda" if torch.cuda.is_available() else "cpu")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
MultiheadAttention-1  [[-1, 120, 128], [-1, 120, 120]]               0
               GRU-2  [[-1, 120, 32], [-1, 2, 16]]               0
               GRU-3  [[-1, 120, 4], [-1, 2, 4]]               0
            Conv1d-4              [-1, 32, 125]           6,176
            Conv1d-5              [-1, 16, 120]           3,088
               GRU-6  [[-1, 120, 8], [-1, 2, 4]]               0
               GRU-7  [[-1, 120, 4], [-1, 2, 4]]               0
            Linear-8                  [-1, 256]         246,016
         LiquidNet-9     [[-1, 128], [-1, 128]]               0
             SiLU-10                  [-1, 128]               0
        KANLinear-11                   [-1, 64]               0
             SiLU-12                   [-1, 64]               0
        KANLinear-13                   [-1, 32]               0
           Linear-14      

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
character_reduction_model.load_state_dict(torch.load('Saved/Model_Character_Reduction_125_64.pth', map_location = device))
model = torch.load('Saved/Model_Combined_Word_Positional_Character.pth', weights_only = False)

In [24]:
# Test the model
test(model, character_reduction_model, dataloader)

  0%|          | 0/5 [00:00<?, ?it/s]

Test Loss: 0.1781, Test Accuracy: 100.00%


## FakeNewsNet -> 80.54%
## LIAR-PLUS -> 54.42%
## ISOT -> 99.65%
## Politifact -> 69.8%
## FakeNewsCorpus -> 100%